In [1]:
import os
#os.chdir("C:\\Users\\Le Tian\\Desktop\\Ensemble modeling\\comp-401")
os.chdir("/Users/gaoletian/Desktop/Ensemble modeling/comp-401")
os.getcwd()

'/Users/gaoletian/Desktop/Ensemble modeling/comp-401'

In [3]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from multi_omics_integration.processing import load_resize
from multi_omics_integration.func import estimators, estimator_parameters

In [4]:
datasets = {
    "rnaseq": "../Datasets/breast/RNASeq.csv"
}
labels = "../Datasets/breast/Clinical_enc.csv"
target_name = "PAM50"

merged_X, X_modalities, y = load_resize(datasets, labels, target_name)

In [5]:
import scanpy as sc
import pandas as pd

# Load RNASeq data
rna = pd.read_csv("../Datasets/breast/RNASeq.csv")

# Set SubjectID as index
rna["SubjectID"] = rna["SubjectID"].astype(str).str.strip()
rna.set_index("SubjectID", inplace=True)

# Convert to AnnData
adata = sc.AnnData(rna)

# Normalize total counts to 1e4
sc.pp.normalize_total(adata, target_sum=1e4)

# Log-transform the data
sc.pp.log1p(adata)

# Identify highly variable genes (HVGs)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')

# Filter to HVGs
adata = adata[:, adata.var['highly_variable']]

# Convert back to pandas DataFrame
rna_processed = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

print("Processed RNA shape:", rna_processed.shape)

Processed RNA shape: (430, 2000)


In [6]:
import pandas as pd

# Convert y to a pandas Series with same index as rna_processed
y = pd.Series(y, index=rna_processed.index)

# Now reassign
merged_X = rna_processed.loc[y.index]

print(" Merged shape:", merged_X.shape)
print(" Target distribution:\n", y.value_counts())


 Merged shape: (430, 2000)
 Target distribution:
 2    229
3    100
0     71
1     30
Name: count, dtype: int64


In [9]:
results = []

for name, model in estimators:
    if name not in estimator_parameters:
        print(f"Skipping '{name}' (no param grid defined)")
        continue

    print(f"\nTuning model: {name}")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", model)
    ])

    param_grid = {f"clf__{k}": v for k, v in estimator_parameters[name].items()}

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=5,
        scoring="accuracy",
        verbose=1,
        n_jobs=-1
    )

    try:
        grid.fit(merged_X, y)
        results.append({
            "model": name,
            "best_score": grid.best_score_,
            "best_params": grid.best_params_
        })
        print(f" Best score for {name}: {grid.best_score_:.4f}")
        print(f" Best parameters for {name}: {grid.best_params_}")
    except Exception as e:
        print(f" Failed to fit {name}: {e}")


Tuning model: logistic
Fitting 5 folds for each of 5 candidates, totalling 25 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in ve

 Best score for logistic: 0.9023
 Best parameters for logistic: {'clf__C': 0.01}

Tuning model: random_forest
Fitting 5 folds for each of 16 candidates, totalling 80 fits
 Best score for random_forest: 0.9070
 Best parameters for random_forest: {'clf__max_depth': 7, 'clf__n_estimators': 200}

Tuning model: deep_nn
Fitting 5 folds for each of 864 candidates, totalling 4320 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_

 Best score for deep_nn: 0.9116
 Best parameters for deep_nn: {'clf__activation': 'relu', 'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (50,), 'clf__learning_rate': 'constant', 'clf__solver': 'lbfgs'}
Skipping 'svc' (no param grid defined)


In [ ]:
results_df = pd.DataFrame(results).sort_values(by="best_score", ascending=False)
print("\nSummary of Results:")
display(results_df)


Summary of Results:


,model,best_score,best_params
2,deep_nn,0.911628,"{'clf__activation': 'relu', 'clf__alpha': 0.00..."
1,random_forest,0.906977,"{'clf__max_depth': 7, 'clf__n_estimators': 200}"
0,logistic,0.902326,{'clf__C': 0.01}
